## Quaterly Sales by Product

In [0]:
# Creating dataframes using silver table: sales_order_items
df_sales_order_items = spark.read.format('delta').table('bikesales.silver.sales_order_items')

In [0]:
# Creating dataframes using silver table: products
df_products = spark.read.format('delta').table('bikesales.silver.products')

In [0]:
# Creating dataframes using silver table: sales_orders
df_sales_orders = spark.read.format('delta').table('bikesales.silver.sales_orders')

In [0]:
# Selecting relevant columns as per the requirment i.e. Quaterly Sales by Product
df_sales_order_items_q1 = df_sales_order_items.select('SALESORDERID', 'PRODUCTID', "QUANTITY", "NETAMOUNT")
df_products_q1 = df_products.select('PRODUCTID', 'PRODCATEGORYID', 'CURRENCY')
df_sales_orders_q1 = df_sales_orders.select("SALESORDERID", 'SALESORG', 'FISCALYEAR', 'FISCALPERIOD')

In [0]:
# Joining df_sales_order_items_q1 and df_products_q1
df_sales_order_items_join1 = df_sales_order_items_q1.join(df_products_q1, df_sales_order_items_q1.PRODUCTID == df_products_q1.PRODUCTID, 'inner').select(df_sales_order_items_q1["*"], df_products_q1.PRODCATEGORYID, df_products_q1.CURRENCY)
# Joining df_sales_order_items_join1 and df_sales_orders_q1
df_sales_order_items_join2 = df_sales_order_items_join1.join(df_sales_orders_q1, df_sales_order_items_join1.SALESORDERID == df_sales_orders_q1.SALESORDERID, 'inner').select(df_sales_order_items_join1['*'], df_sales_orders_q1.SALESORG, df_sales_orders_q1.FISCALYEAR, df_sales_orders_q1.FISCALPERIOD)

In [0]:
# Grouping the records on PRODUCTID, FISCALPERIOD, FISCALYEAR and perform aggregation to get business relevant information as per requirement
from pyspark.sql import functions as F
df_sales_order_items_agg = df_sales_order_items_join2.groupBy('PRODUCTID', 'FISCALPERIOD', 'FISCALYEAR').agg(F.count('PRODUCTID').alias('total_orders'), F.sum('QUANTITY').alias('total_units_sold'), F.sum('NETAMOUNT').alias('total_revenue'))

In [0]:
# Joining the results to form the dataset
df_Quaterly_Sales_by_Product = (
    df_sales_order_items_join2.alias('sales_order_items').join(df_sales_order_items_agg.alias('sales_agg'),
                                                               (F.col("sales_order_items.PRODUCTID") == F.col("sales_agg.PRODUCTID")) &
                                                               (F.col("sales_order_items.FISCALPERIOD")== F.col("sales_agg.FISCALPERIOD")) &    
                                                               (F.col("sales_order_items.FISCALYEAR")== F.col("sales_agg.FISCALYEAR"))
                                                               ,
                                                               'inner')).select(df_sales_order_items_join2['*'],df_sales_order_items_agg.total_orders, df_sales_order_items_agg.total_units_sold,
                                                               df_sales_order_items_agg.total_revenue)

In [0]:
# Creating gold dataset to get Quaterly Sales by Product
df_Quaterly_Sales_by_Product.write.format('delta').mode('overwrite').saveAsTable('bikesales.gold.Quaterly_Sales_by_Product')